# Laboratorio 9 Tareas 3 y 4

In [ ]:
import json, math, os, random, time
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(min(8, os.cpu_count() or 1))
AMP_ENABLED=torch.cuda.is_available()
if AMP_ENABLED: torch.backends.cudnn.benchmark=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path('models').mkdir(exist_ok=True); Path('figs').mkdir(exist_ok=True)
CLASSES=['T-shirt/top','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']
raw=datasets.FashionMNIST('data',train=True,download=True)
x_all=raw.data.float().div(255).unsqueeze(1); y_all=raw.targets.long()
x_train,x_val=x_all[:-5000],x_all[-5000:]
y_train,y_val=y_all[:-5000],y_all[-5000:]
# Fashion-MNIST cabe normalmente en GPU: evita 30 * 430 transferencias por lote.
# Si no cabe, conserva la copia CPU y transfiere cada lote como antes.
if device.type=='cuda':
    try:
        x_train_gpu=x_train.to(device,non_blocking=True)*2-1
        x_val_gpu=x_val.to(device,non_blocking=True)*2-1
        y_train_gpu=y_train.to(device);y_val_gpu=y_val.to(device)
        DATA_ON_GPU=True
    except torch.cuda.OutOfMemoryError:
        for name in ('x_train_gpu','x_val_gpu','y_train_gpu','y_val_gpu'):
            if name in globals():del globals()[name]
        torch.cuda.empty_cache();DATA_ON_GPU=False
else: DATA_ON_GPU=False
print('GPU:',torch.cuda.get_device_name(0) if AMP_ENABLED else 'no disponible',
      '| datos en GPU:',DATA_ON_GPU,'| precisión mixta:',AMP_ENABLED)
T=1000
betas=torch.zeros(T+1); betas[1:]=torch.linspace(1e-4,.02,T)
alphas=1-betas; abar=alphas.cumprod(0)
betas_d,alphas_d,abar_d=[a.to(device) for a in (betas,alphas,abar)]
def q_sample(x0,t,eps):
    # x_t = sqrt(abar_t) x_0 + sqrt(1-abar_t) epsilon
    a=abar_d[t].view(-1,1,1,1)
    return a.sqrt()*x0+(1-a).sqrt()*eps
def batches(n,size,shuffle=True):
    ids=torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0,n,size):yield ids[i:i+size]
def figsave(name):
    plt.savefig('figs/'+name,dpi=170,bbox_inches='tight');plt.close()
def mosaic(imgs,rows,cols):
    imgs=imgs.detach().cpu().view(-1,28,28).numpy()
    canvas=np.ones((rows*29+1,cols*29+1),np.float32)
    for i,img in enumerate(imgs[:rows*cols]):
        r,c=divmod(i,cols);canvas[1+r*29:29+r*29,1+c*29:29+c*29]=img
    return canvas
def sharpness(imgs):return (imgs[...,1:]-imgs[...,:-1]).abs().mean().item()



GPU: NVIDIA GeForce RTX 3050 Laptop GPU | datos en GPU: True | precisión mixta: True


## Tarea 3.1 U Net condicional
PE(t,2i)=sin(t/10000^(2i/d)); PE(t,2i+1)=cos(t/10000^(2i/d)).
Tiempo y clase se suman y se inyectan por broadcasting en cada bloque.



In [ ]:
def sinusoidal(t,d=64):
    freq=torch.exp(-math.log(10000)*torch.arange(d//2,device=t.device).float()/(d//2))
    a=t.float()[:,None]*freq[None,:]
    return torch.cat((a.sin(),a.cos()),1)
# Tabla de embeddings de tiempo: se usa también 1000 veces en cada muestreo.
TIME_FEATURES=sinusoidal(torch.arange(T+1,device=device))

class Block(nn.Module):
    def __init__(self,ci,co):
        super().__init__()
        self.conv1=nn.Conv2d(ci,co,3,padding=1);self.conv2=nn.Conv2d(co,co,3,padding=1)
        self.norm1=nn.GroupNorm(8,co);self.norm2=nn.GroupNorm(8,co)
        self.embed=nn.Linear(128,co)
        self.skip=nn.Conv2d(ci,co,1) if ci!=co else nn.Identity()
    def forward(self,x,e):
        h=F.silu(self.norm1(self.conv1(x)))+self.embed(e)[:,:,None,None]
        return F.silu(self.norm2(self.conv2(F.silu(h)))+self.skip(x))
class ConditionalUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.t_mlp=nn.Sequential(nn.Linear(64,128),nn.SiLU(),nn.Linear(128,128))
        self.c_embed=nn.Embedding(11,128)
        self.stem=nn.Conv2d(1,32,3,padding=1);self.enc28=Block(32,32)
        self.down=nn.Conv2d(32,64,3,stride=2,padding=1);self.enc14=Block(64,64)
        self.mid=Block(64,64);self.up=nn.ConvTranspose2d(64,64,4,stride=2,padding=1)
        self.dec28=Block(96,32);self.out=nn.Conv2d(32,1,3,padding=1)
    def forward(self,x,t,c):
        e=self.t_mlp(TIME_FEATURES[t])+self.c_embed(c)
        s=self.enc28(self.stem(x),e)
        h=self.mid(self.enc14(self.down(s),e),e)
        return self.out(self.dec28(torch.cat((self.up(h),s),1),e))
unet=ConditionalUNet().to(device)
parameters=sum(p.numel() for p in unet.parameters())
assert unet(torch.zeros(2,1,28,28,device=device),torch.tensor([1,1000],device=device),torch.tensor([0,10],device=device)).shape==(2,1,28,28)
print('Parámetros:',parameters)



Parámetros: 342721


## Tarea 3.2 Entrenamiento y validación
30 épocas, lotes de 128, t uniforme por imagen, probabilidad .1 de condición nula.
Los puntos de control permiten reanudar tras una época completa.



In [ ]:
checkpoint=Path('models/diffusion_last.pt')
optimizer=torch.optim.Adam(unet.parameters(),lr=2e-4)
scaler=torch.amp.GradScaler('cuda',enabled=AMP_ENABLED)
history=[];start=0
if checkpoint.exists():
    state=torch.load(checkpoint,map_location=device,weights_only=False)
    unet.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
    history=state['history'];start=state['epoch']
    if 'scaler' in state:scaler.load_state_dict(state['scaler'])
    # map_location=device también mueve el estado RNG de CPU a CUDA; devolverlo a CPU.
    torch.set_rng_state(state['rng'].cpu())
    if device.type=='cuda' and state.get('cuda_rng') is not None:
        torch.cuda.set_rng_state_all([rng.cpu() for rng in state['cuda_rng']])
for epoch in range(start,30):
    unet.train();total=torch.zeros((),device=device)
    for ids in batches(len(x_train),128):
        if DATA_ON_GPU:
            idx=ids.to(device,non_blocking=True);x=x_train_gpu[idx];c=y_train_gpu[idx].clone()
        else:x=x_train[ids].to(device)*2-1;c=y_train[ids].to(device).clone()
        t=torch.randint(1,T+1,(len(ids),),device=device)
        eps=torch.randn_like(x)
        c[torch.rand(len(ids),device=device)<.1]=10
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=AMP_ENABLED):
            loss=F.mse_loss(unet(q_sample(x,t,eps),t,c),eps)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
        total+=loss.detach()*len(ids)
    history.append((total/len(x_train)).item())
    torch.save({'epoch':epoch+1,'model':unet.state_dict(),'optimizer':optimizer.state_dict(),
                'history':history,'scaler':scaler.state_dict(),'rng':torch.get_rng_state(),
                'cuda_rng':torch.cuda.get_rng_state_all() if device.type=='cuda' else None},checkpoint)
    print('Época',epoch+1,'MSE',history[-1],flush=True)
unet.eval()
plt.figure(figsize=(7,3.5));plt.plot(range(1,31),history)
plt.xlabel('Época');plt.ylabel('MSE de entrenamiento');figsave('t3_2_entrenamiento.png')
interval_losses=[]
with torch.inference_mode():
    for interval in range(10):
        total=0.
        for ids in batches(len(x_val),128,False):
            if DATA_ON_GPU:
                idx=ids.to(device,non_blocking=True);x=x_val_gpu[idx];c=y_val_gpu[idx]
            else:x=x_val[ids].to(device)*2-1;c=y_val[ids].to(device)
            t=torch.randint(100*interval+1,100*interval+101,(len(ids),),device=device)
            eps=torch.randn_like(x)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=AMP_ENABLED):
                pred=unet(q_sample(x,t,eps),t,c)
            total+=F.mse_loss(pred.float(),eps,reduction='sum').item()
        interval_losses.append(total/(len(x_val)*784))
plt.figure(figsize=(8,3.5));plt.bar(range(1,11),interval_losses)
plt.xlabel('Intervalo de 100 pasos');plt.ylabel('MSE de validación');figsave('t3_2_validacion_intervalos.png')



Época 27 MSE 0.04201342537999153
Época 28 MSE 0.04136251285672188
Época 29 MSE 0.04090072214603424
Época 30 MSE 0.040203746408224106


## Tareas 3.3 y 3.4 Muestreo y guía
epsilon_tilde=epsilon_uncond+w(epsilon_cond-epsilon_uncond).
x_(t-1)=(x_t-beta_t epsilon_tilde/sqrt(1-abar_t))/sqrt(alpha_t)+sqrt(beta_t) z;
z=0 al terminar. Sólo se recorta la imagen final al intervalo [-1,1].



In [ ]:
@torch.inference_mode()
def sample(classes,w,generator):
    classes=classes.to(device)
    x=torch.randn((len(classes),1,28,28),device=device,generator=generator)
    with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=AMP_ENABLED):
        for step in range(T,0,-1):
            t=torch.full_like(classes,step)
            if w==1:pred=unet(x,t,classes)
            else:
                both=unet(torch.cat((x,x)),torch.cat((t,t)),
                          torch.cat((torch.full_like(classes,10),classes)))
                a,b=both.chunk(2);pred=a+w*(b-a)
            x=(x-betas_d[step]*pred/(1-abar_d[step]).sqrt())/alphas_d[step].sqrt()
            if step>1:x=x+betas_d[step].sqrt()*torch.randn(x.shape,device=device,generator=generator)
    return x.clamp(-1,1).cpu()
def generate(classes,w,seed,batch_size=100):
    final=Path(f'models/generated_w{w}.pt')
    progress=Path(f'models/generation_w{w}_progress.pt')
    timings_path=Path(f'models/generation_w{w}_seconds.json')
    if final.exists():
        images=torch.load(final,map_location='cpu',weights_only=True)
        if len(images)==len(classes):
            if timings_path.exists():elapsed=json.loads(timings_path.read_text())['seconds']
            else:
                # Archivo creado con la versión anterior: medir 100 y reutilizar las 500.
                g=torch.Generator(device=device).manual_seed(seed)
                if AMP_ENABLED:torch.cuda.synchronize()
                tic=time.perf_counter();sample(classes[:batch_size],w,g)
                if AMP_ENABLED:torch.cuda.synchronize()
                elapsed=(time.perf_counter()-tic)*(len(classes)/batch_size)
                timings_path.write_text(json.dumps({'seconds':elapsed,'estimated_from_100':True}))
            print('Reutilizadas',len(images),'imágenes de w =',w)
            return images,elapsed
    state=torch.load(progress,map_location='cpu',weights_only=False) if progress.exists() else {'chunks':[],'times':[]}
    for k in range(len(state['chunks'])*batch_size,len(classes),batch_size):
        g=torch.Generator(device=device).manual_seed(seed+k)
        if AMP_ENABLED:torch.cuda.synchronize()
        tic=time.perf_counter();chunk=sample(classes[k:k+batch_size],w,g)
        if AMP_ENABLED:torch.cuda.synchronize()
        state['chunks'].append(chunk);state['times'].append(time.perf_counter()-tic)
        torch.save(state,progress)  # permite continuar tras desconexión en Colab
        print('w',w,'imágenes listas',min(k+batch_size,len(classes)),'/',len(classes),flush=True)
    images=torch.cat(state['chunks']);elapsed=sum(state['times'])
    torch.save(images,final)
    timings_path.write_text(json.dumps({'seconds':elapsed,'estimated_from_100':False}))
    progress.unlink(missing_ok=True)
    return images,elapsed
requested=torch.arange(10).repeat_interleave(50)
generated={};timings={}
for w in (1,3,7):
    generated[w],timings[w]=generate(requested,w,SEED+321)
    print('w',w,'segundos',timings[w])
indices=(torch.arange(10)[:,None]*50+torch.arange(10)[None,:]).flatten()
plt.figure(figsize=(10,10));plt.imshow(mosaic((generated[1][indices]+1)/2,10,10),cmap='gray',vmin=0,vmax=1)
plt.axis('off');figsave('t3_3_muestras_por_clase.png')
plt.figure(figsize=(10,3.5))
examples=torch.cat([generated[w][450:460] for w in (1,3,7)])
plt.imshow(mosaic((examples+1)/2,3,10),cmap='gray',vmin=0,vmax=1)
plt.axis('off');plt.title('Ankle boot: filas w=1,3,7');figsave('t3_4_guia_comparacion.png')



w 1 imágenes listas 100 / 500
w 1 imágenes listas 200 / 500
w 1 imágenes listas 300 / 500
w 1 imágenes listas 400 / 500
w 1 imágenes listas 500 / 500
w 1 segundos 55.522252400027355
w 3 imágenes listas 100 / 500
w 3 imágenes listas 200 / 500
w 3 imágenes listas 300 / 500
w 3 imágenes listas 400 / 500
w 3 imágenes listas 500 / 500
w 3 segundos 103.45770720002474
w 7 imágenes listas 100 / 500
w 7 imágenes listas 200 / 500
w 7 imágenes listas 300 / 500
w 7 imágenes listas 400 / 500
w 7 imágenes listas 500 / 500
w 7 segundos 101.11495700001251


## Tarea 3.4 Clasificador, fidelidad y diversidad
Distancia L2 media entre los 1225 pares de cada clase. Referencia: 50 imágenes
reales de validación por clase. Si no llega al 88 por ciento, se detiene.



In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),
            nn.Linear(64*7*7,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x):return self.net(x)
clf=Classifier().to(device);clf_file=Path('models/classifier.pt')
if clf_file.exists():clf.load_state_dict(torch.load(clf_file,map_location=device,weights_only=True))
else:
    opt=torch.optim.Adam(clf.parameters(),lr=1e-3)
    for epoch in range(20):
        clf.train()
        for ids in batches(len(x_train),256):
            loss=F.cross_entropy(clf(x_train[ids].to(device)),y_train[ids].to(device))
            opt.zero_grad(set_to_none=True);loss.backward();opt.step()
        clf.eval()
        with torch.inference_mode():
            predictions=torch.cat([clf(x_val[ids].to(device)).argmax(1).cpu()
                                   for ids in batches(len(x_val),256,False)])
        print('Clasificador época',epoch+1,'exactitud', (predictions==y_val).float().mean().item())
        if (predictions==y_val).float().mean().item()>=.88:break
    torch.save(clf.state_dict(),clf_file)
clf.eval()
with torch.inference_mode():
    val_predictions=torch.cat([clf(x_val[ids].to(device)).argmax(1).cpu()
                               for ids in batches(len(x_val),256,False)])
accuracy=(val_predictions==y_val).float().mean().item()
assert accuracy>=.88,f'Exactitud insuficiente: {accuracy:.3%}'
@torch.inference_mode()
def classify(imgs):
    return torch.cat([clf(((imgs[i:i+128]+1)/2).to(device)).argmax(1).cpu()
                      for i in range(0,len(imgs),128)])
def diversity(imgs):return torch.pdist(imgs.float().flatten(1),p=2).mean().item()
fidelity={};diversities={};preds={}
for w in (1,3,7):
    preds[w]=classify(generated[w]);fidelity[w]=(preds[w]==requested).float().mean().item()
    diversities[w]=[diversity(generated[w][50*c:50*(c+1)]) for c in range(10)]
real50=torch.cat([x_val[(y_val==c).nonzero().flatten()[:50]] for c in range(10)])
assert len(real50)==500
real_diversity=[diversity(real50[50*c:50*(c+1)]*2-1) for c in range(10)]
confusion=torch.zeros(10,10,dtype=torch.long)
for target,pred in zip(requested,preds[1]):confusion[target,pred]+=1
wrong=confusion.clone();wrong.fill_diagonal_(0)
most_confused=torch.nonzero(torch.triu(wrong+wrong.T,diagonal=1)==torch.triu(wrong+wrong.T,diagonal=1).max())[0].tolist()
plt.figure(figsize=(7,6));plt.imshow(confusion,cmap='Blues')
plt.xticks(range(10),CLASSES,rotation=55,ha='right',fontsize=8);plt.yticks(range(10),CLASSES,fontsize=8)
plt.xlabel('Predicha');plt.ylabel('Solicitada');plt.colorbar();figsave('t3_5_matriz_confusion.png')
for w in (1,3,7):print(w,'fidelidad',fidelity[w],'diversidad',np.mean(diversities[w]),'s/100',timings[w]/5)
print('Exactitud clasificador',accuracy,'referencia diversidad',np.mean(real_diversity),'par confundido',most_confused)



Clasificador época 1 exactitud 0.8611999750137329
Clasificador época 2 exactitud 0.8737999796867371
Clasificador época 3 exactitud 0.8855999708175659
1 fidelidad 0.7820000052452087 diversidad 15.054572010040284 s/100 11.104450480005472
3 fidelidad 0.9919999837875366 diversidad 13.373748397827148 s/100 20.691541440004947
7 fidelidad 0.9959999918937683 diversidad 12.701724815368653 s/100 20.222991400002503
Exactitud clasificador 0.8855999708175659 referencia diversidad 17.675182342529297 par confundido [4, 6]


## Tarea 4.1 Nitidez, rapidez y evaluaciones
Comparar imágenes en la misma escala [0,1]. Diffusion w=1: 1000 evaluaciones
de U Net por imagen; VAE: una evaluación de decoder.



In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(784,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU())
        self.fc_mu=nn.Linear(256,2);self.fc_logvar=nn.Linear(256,2)
        self.dec=nn.Sequential(nn.Linear(2,256),nn.ReLU(),nn.Linear(256,512),nn.ReLU(),nn.Linear(512,784),nn.Sigmoid())
    def decode(self,z):return self.dec(z)
    def forward(self,x):
        h=self.enc(x);mu=self.fc_mu(h);lv=self.fc_logvar(h)
        return self.decode(mu+torch.exp(.5*lv)*torch.randn_like(mu)),mu,lv
vae=VAE().to(device)
vae.load_state_dict(torch.load('models/vae_beta1.0.pt',map_location=device,weights_only=True))
vae.eval()
with torch.inference_mode():
    vae500=vae.decode(torch.randn(500,2,device=device)).cpu().view(-1,1,28,28)
    z100=torch.randn(100,2,device=device)
    if device.type=='cuda':torch.cuda.synchronize()
    tic=time.perf_counter();vae.decode(z100)
    if device.type=='cuda':torch.cuda.synchronize()
    vae_seconds100=time.perf_counter()-tic
sharp={'real':sharpness(real50),'diffusion_w1':sharpness((generated[1]+1)/2),
       'vae_beta1':sharpness(vae500)}
ratio=(timings[1]/5)/vae_seconds100
print('Nitidez',sharp,'segundos VAE/100',vae_seconds100,'difusión/100',timings[1]/5,'razón',ratio)



Nitidez {'real': 0.08861149102449417, 'diffusion_w1': 0.08538998663425446, 'vae_beta1': 0.04982313513755798} segundos VAE/100 0.001132100005634129 difusión/100 11.104450480005472 razón 9808.718686283795


## Tarea 4.2 Estimador pass@k
Sin reemplazo: 1 - producto_i((n-c-i)/(n-i)); si n-c<k, pass@k=1.



In [ ]:
def pass_at_k(n,c,k):
    if not 1<=k<=n:raise ValueError('k fuera de 1..n')
    if n-c<k:return 1.
    failure=1.
    for i in range(k):failure*=(n-c-i)/(n-i)
    return 1-failure
pass_rows=[]
for name,c in zip(('P1','P2','P3','P4','P5','P6'),(0,2,5,9,14,20)):
    pass_rows.append({'problema':name,'c':c,**{f'pass@{k}':pass_at_k(20,c,k) for k in (1,5,10)},
                      'ingenuo@5':1-(1-c/20)**5})
pass_means={f'pass@{k}':sum(row[f'pass@{k}'] for row in pass_rows)/6 for k in (1,5,10)}
print(pass_rows,pass_means,'P2 mínimo k > .9:',next(k for k in range(1,21) if pass_at_k(20,2,k)>.9))



[{'problema': 'P1', 'c': 0, 'pass@1': 0.0, 'pass@5': 0.0, 'pass@10': 0.0, 'ingenuo@5': 0.0}, {'problema': 'P2', 'c': 2, 'pass@1': 0.09999999999999998, 'pass@5': 0.44736842105263164, 'pass@10': 0.763157894736842, 'ingenuo@5': 0.40950999999999993}, {'problema': 'P3', 'c': 5, 'pass@1': 0.25, 'pass@5': 0.8063080495356038, 'pass@10': 0.9837461300309598, 'ingenuo@5': 0.7626953125}, {'problema': 'P4', 'c': 9, 'pass@1': 0.44999999999999996, 'pass@5': 0.9702012383900929, 'pass@10': 0.9999404620147654, 'ingenuo@5': 0.9496715625}, {'problema': 'P5', 'c': 14, 'pass@1': 0.7, 'pass@5': 0.9996130030959752, 'pass@10': 1.0, 'ingenuo@5': 0.99757}, {'problema': 'P6', 'c': 20, 'pass@1': 1.0, 'pass@5': 1.0, 'pass@10': 1.0, 'ingenuo@5': 1.0}] {'pass@1': 0.4166666666666667, 'pass@5': 0.7039151186790505, 'pass@10': 0.7911407477970945} P2 mínimo k > .9: 14


## Resultados para completar el informe después de ejecutar las 30 épocas



In [ ]:
results={'seed':SEED,'device':str(device),'parameters':parameters,
 'train_mse_by_epoch':history,'validation_mse_by_t_interval':interval_losses,
 'classifier_validation_accuracy':accuracy,
 'fidelity':{str(w):fidelity[w] for w in (1,3,7)},
 'diversity_by_class':{str(w):diversities[w] for w in (1,3,7)},
 'real_diversity_by_class':real_diversity,
 'seconds_per_100':{str(w):timings[w]/5 for w in (1,3,7)},
 'confusion_matrix_w1':confusion.tolist(),'most_confused_pair':most_confused,
 'sharpness_500':sharp,'vae_seconds_per_100':vae_seconds100,
 'diffusion_to_vae_time_ratio':ratio,'pass_at_k':pass_rows,'pass_means':pass_means}
with open('results_task3_task4.json','w',encoding='utf-8') as f:
    json.dump(results,f,indent=2,ensure_ascii=False)
print('Archivo results_task3_task4.json guardado')


Archivo results_task3_task4.json guardado
